# 1. Thiết lập Môi trường & Kiểm tra Phần cứng (T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NhatMinh1503/pruning-resnet18/blob/benchmark/benchmark/colab_benchmark.ipynb)

Notebook đo lường chính thức hiệu năng (FLOPs, Dung lượng, Latency CPU/GPU đa batch size) cho Q5 (Structured Pruning) và các checkpoint đại diện trên Colab T4.

In [ ]:
import os
import sys
import torch
from google.colab import drive

# 1. Kết nối Google Drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/pruning-project'
CKPT = os.path.join(DRIVE_ROOT, 'checkpoints')
RESULTS = os.path.join(DRIVE_ROOT, 'results')

# 2. Kiểm tra tồn tại thư mục checkpoints (hỗ trợ tài khoản phụ dùng Shortcut)
assert os.path.exists(CKPT), (
    f"⚠️ Không tìm thấy '{CKPT}'!\n"
    "Nếu bạn dùng tài khoản phụ được chia sẻ thư mục 'pruning-project', "
    "hãy vào mục 'Được chia sẻ với tôi' (Shared with me) trên Google Drive, "
    "click chuột phải vào 'pruning-project' -> chọn 'Thêm lối tắt vào Drive' (Add shortcut to Drive) -> chọn 'MyDrive'."
)

# 3. Kiểm tra GPU
assert torch.cuda.is_available(), (
    "⚠️ KHÔNG CÓ GPU!\n"
    "Vui lòng vào menu: Runtime -> Change runtime type -> Hardware accelerator chọn T4 GPU."
)

# 4. In thông tin phần cứng
print("=== THÔNG TIN PHẦN CỨNG ===")
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!lscpu | grep -E "Model name|CPU\(s\):"
print(f"PyTorch version: {torch.__version__} | CUDA version: {torch.version.cuda}")

# 5. Lấy code mới nhất từ GitHub và cài đặt thư viện
REPO_URL = "https://github.com/NhatMinh1503/pruning-resnet18.git"
REPO_DIR = "/content/pruning-resnet18"

if os.path.isdir(REPO_DIR):
    %cd {REPO_DIR}
    !git fetch origin && git checkout main && git pull origin main
else:
    !git clone -b main {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

!pip install -q thop
print("\n[OK] Thiết lập môi trường hoàn tất và đã chuyển vào thư mục repo.")

# 2. Kiểm tra Checkpoints trên Drive & Dữ liệu INT8 Sơ bộ của Khang

In [ ]:
import os

print("=== DANH SÁCH CHECKPOINT STRUCTURED (Q5) TRÊN DRIVE ===")
!ls -lh {CKPT}/resnet18_structured_*.pt

all_pt = [f for f in os.listdir(CKPT) if f.endswith('.pt')]
print(f"\nTổng số file checkpoint (.pt) trong Drive: {len(all_pt)}")

int8_csv_drive = os.path.join(RESULTS, 'int8_efficiency.csv')
print(f"\n=== KIỂM ĐỊNH FILE INT8 TRÊN DRIVE ({int8_csv_drive}) ===")
if os.path.exists(int8_csv_drive):
    print(f"-> Đã tìm thấy {int8_csv_drive}, nội dung:")
    with open(int8_csv_drive, 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print("-> Chưa có file int8_efficiency.csv trên Drive chung (sẽ được cập nhật sau khi chạy benchmark INT8 chính thức).")

# 3. Đo đạc Hiệu năng Q5 Đa Batch Size (1, 16, 64) trên 11 Checkpoint Đại diện

In [ ]:
import os
import subprocess
import sys

out_csv = "results/structured_efficiency.csv"
if os.path.exists(out_csv):
    os.remove(out_csv)
    print(f"Đã xóa file cũ '{out_csv}' để đo sạch trong 1 phiên liên tục.")

checkpoints = [
    "resnet18_baseline.pt",
    "resnet18_local_0.5_finetuned.pt",
    "resnet18_local_0.9_finetuned.pt",
    "resnet18_global_0.5_finetuned.pt",
    "resnet18_global_0.9_finetuned.pt",
    "resnet18_structured_0.25_raw.pt",
    "resnet18_structured_0.25_finetuned.pt",
    "resnet18_structured_0.5_raw.pt",
    "resnet18_structured_0.5_finetuned.pt",
    "resnet18_structured_0.75_raw.pt",
    "resnet18_structured_0.75_finetuned.pt",
]

print(f"Bắt đầu đo đạc {len(checkpoints)} checkpoints đại diện (Batch sizes: 1, 16, 64)...\n")
for ckpt_name in checkpoints:
    ckpt_path = os.path.join(CKPT, ckpt_name)
    if not os.path.exists(ckpt_path):
        print(f"\n[WARNING] Không tìm thấy: {ckpt_path} -> Bỏ qua và tiếp tục...")
        continue

    print(f"\n=======================================================")
    print(f"-> Đang đo: {ckpt_name}")
    print(f"=======================================================")
    cmd = [
        sys.executable, "-u", "benchmark/measure.py",
        "--checkpoint", ckpt_path,
        "--batch-sizes", "1", "16", "64",
        "--output-csv", out_csv
    ]
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    assert proc.wait() == 0, f"Lỗi khi đo benchmark: {ckpt_name}"

print(f"\n[HOÀN TẤT] Quá trình benchmark 11 checkpoint đã hoàn thành.")

# 4. Xuất Dữ liệu An toàn ra Drive & In Bảng Kết quả

In [ ]:
import os
import shutil

src_csv = "results/structured_efficiency.csv"
assert os.path.exists(src_csv), f"Không tìm thấy file kết quả {src_csv}!"

# (a) Luôn lưu 1 bản sao dự phòng ra MyDrive cá nhân của tài khoản đang chạy
personal_backup = "/content/drive/MyDrive/structured_efficiency.csv"
shutil.copy(src_csv, personal_backup)
print(f"[THÀNH CÔNG] Đã lưu 1 bản sao dự phòng tại MyDrive cá nhân: {personal_backup}")

# (b) Thử copy vào thư mục results/ của Drive chung (an toàn nếu chỉ có quyền Viewer)
shared_target = os.path.join(RESULTS, "structured_efficiency.csv")
try:
    shutil.copy(src_csv, shared_target)
    print(f"[THÀNH CÔNG] Đã đồng bộ kết quả vào Drive chung: {shared_target}")
except OSError as e:
    print(f"[CẢNH BÁO] Không thể ghi vào Drive chung (có thể do tài khoản chỉ có quyền Viewer): {e}")
    print(f"-> File kết quả vẫn được bảo toàn an toàn tại MyDrive cá nhân của bạn ({personal_backup}).")

# In toàn bộ nội dung file kết quả ra màn hình để sao chép / kiểm định
print(f"\n=== TOÀN BỘ NỘI DUNG FILE {src_csv} ===")
with open(src_csv, "r", encoding="utf-8") as f:
    print(f.read())